# ACE PDF Ingestion — Production

Parses lending policy PDFs from the Unity Catalog volume, chunks them, writes to `policy_documents`, and triggers a Vector Search index sync.

Set `WIPE_BEFORE_INGEST = True` in Cell 2 for a full reload.
Set to `False` to append new documents without clearing existing chunks.

| Cell | Purpose |
|------|---------|
| 1 | Install `pypdf` and `databricks-vectorsearch` |
| 2 | Central config — `VOLUME_PATH`, chunking settings, `WIPE_BEFORE_INGEST` flag |
| 3 | Clear existing chunks (controlled by `WIPE_BEFORE_INGEST`) |
| 4 | Parse, chunk, and write all PDFs to Delta |
| 5 | Trigger Vector Search index sync |

> **Prod note:** Volume validation and verification cells removed. Confirm `VOLUME_PATH` in Cell 2 is correct before running.


In [ ]:
# ================================================================
# CELL 1 — Install Dependencies
# ================================================================
#
# PURPOSE:
#   Installs the two packages this notebook needs before any other
#   cell can run.
#
# PACKAGES:
#   pypdf
#     The PDF text extraction library. Used in Cell 5 (process_pdf)
#     to read each page of a policy document and extract its raw text.
#     It handles multi-page PDFs, embedded fonts, and page-by-page
#     extraction. It does NOT perform OCR — if a PDF contains only
#     scanned images with no embedded text layer, pypdf will return
#     empty strings and the document will produce zero chunks.
#     In that case, the PDF must be OCR-processed before ingestion.
#
#   databricks-vectorsearch
#     Provides VectorSearchClient, used in Cell 6 to trigger the
#     index sync after writing chunks to Delta, and in Cell 7 to run
#     a verification similarity query. The same client is used in
#     01_infrastructure to create the endpoint and index.
#
# KERNEL RESTART:
#   `dbutils.library.restartPython()` forces the Python kernel to
#   restart so the newly installed packages are loaded into the
#   interpreter. After the restart, run from Cell 2 — do NOT re-run
#   Cell 1.
# ================================================================

%pip install pypdf databricks-vectorsearch
dbutils.library.restartPython()

In [ ]:
# ================================================================
# CELL 2 — Central Configuration
# ================================================================
#
# PURPOSE:
#   All paths, resource names, and tuning parameters live here.
#   No value is hardcoded in the cells below — edit only this cell
#   when anything needs to change.
#
# VARIABLE REFERENCE:
#
# ── Source ────────────────────────────────────────────────────────
#   VOLUME_PATH
#     Path to the Unity Catalog volume that holds the policy PDFs.
#     Format: /Volumes/<catalog>/<schema>/<volume_name>
#     All .pdf files at this path are processed. Subdirectories are
#     not traversed — place all PDFs directly in this folder.
#
# ── Destination ───────────────────────────────────────────────────
#   CATALOG / SCHEMA / TABLE_POLICY
#     Unity Catalog location of the destination Delta table.
#     Must match 01_infrastructure — the table is not created here.
#
# ── Vector Search ─────────────────────────────────────────────────
#   VS_ENDPOINT_NAME / VS_INDEX_NAME
#     The VS endpoint and index created in 01_infrastructure.
#     Used in Cell 6 (trigger sync) and Cell 7 (verification query).
#     Must match 01_infrastructure exactly.
#
# ── Chunking Tuning ───────────────────────────────────────────────
#   MAX_CHUNK_CHARS (default: 1200)
#     Maximum characters per chunk. 1200 is tuned for lending policy
#     text — long enough to hold a full clause or sub-section, short
#     enough for precise retrieval. The RAG agent fetches the top 7
#     chunks per query; if chunks are too large, a single chunk may
#     dominate the context window. Tune if retrieval quality suffers:
#       Lower (e.g. 800)  → more granular, better for short Q&A
#       Higher (e.g. 1500) → more context per chunk, better for
#                            questions that span multiple sentences
#
#   MIN_CHUNK_CHARS (default: 30)
#     Chunks shorter than this are silently discarded.
#     Catches PDF extraction artifacts: page numbers, running headers,
#     footers, watermarks, and lone punctuation. Raise this value if
#     you're seeing garbage rows in the policy_documents table.
#
#   HEADING_MAX_LENGTH (default: 80)
#     Maximum characters captured from a detected heading line.
#     Prevents runaway regex matches from treating long lines as
#     headings and consuming body text into the section label.
#
# ── Embedding Model ───────────────────────────────────────────────
#   EMBEDDING_MODEL
#     Must match the model specified in 01_infrastructure Cell 8.
#     Changing this here has no effect — the model is baked into
#     the VS index at creation time. Only change if recreating the
#     index from scratch.
#
# ── Ingestion Behaviour ───────────────────────────────────────────
#   WIPE_BEFORE_INGEST (default: True)
#     True  — DELETE all rows from policy_documents before ingesting.
#             Recommended for most runs. Ensures no stale chunks from
#             old PDF versions remain after an update.
#     False — Append only. Safe when adding brand-new documents that
#             have never been ingested. Risky when updating existing
#             PDFs — old chunks from the previous version will persist.
#
#   SYNC_AFTER_INGEST (default: True)
#     True  — Trigger a VS index sync after the Delta write. Required
#             for the RAG agent to see the new chunks. The sync reads
#             the Change Data Feed and re-embeds only changed rows.
#     False — Skip sync. Use when batching multiple ingestion runs
#             and triggering sync manually once at the end.
# ================================================================

# ── Source ──────────────────────────────────────────────────────
VOLUME_PATH          = "/Volumes/ace_theorem/chat/pdf_assets"

# ── Destination ─────────────────────────────────────────────────
CATALOG              = "ace_theorem"
SCHEMA               = "chat"
TABLE_POLICY         = f"{CATALOG}.{SCHEMA}.policy_documents"

# ── Vector Search ───────────────────────────────────────────────
VS_ENDPOINT_NAME     = "ace-chat-vs-endpoint"
VS_INDEX_NAME        = f"{CATALOG}.{SCHEMA}.policy_documents_index"

# ── Chunking settings ───────────────────────────────────────────
MAX_CHUNK_CHARS      = 1200   # Max characters per chunk before splitting
MIN_CHUNK_CHARS      = 30     # Chunks shorter than this are discarded as noise
HEADING_MAX_LENGTH   = 80     # Max chars captured from a detected heading line

# ── Embedding model (must match 01_infrastructure) ──────────────
EMBEDDING_MODEL      = "databricks-gte-large-en"

# ── Ingestion behaviour flags ───────────────────────────────────
WIPE_BEFORE_INGEST   = True   # Delete all rows before ingesting
SYNC_AFTER_INGEST    = True   # Trigger VS index sync after write

# ── Print summary ───────────────────────────────────────────────
print("=" * 55)
print("ACE PDF Ingestion — Config")
print("=" * 55)
print(f"  Volume path        : {VOLUME_PATH}")
print(f"  Target table       : {TABLE_POLICY}")
print(f"  VS endpoint        : {VS_ENDPOINT_NAME}")
print(f"  VS index           : {VS_INDEX_NAME}")
print(f"  Embedding model    : {EMBEDDING_MODEL}")
print(f"  Max chunk chars    : {MAX_CHUNK_CHARS}")
print(f"  Min chunk chars    : {MIN_CHUNK_CHARS}")
print(f"  Heading max length : {HEADING_MAX_LENGTH}")
print(f"  Wipe before ingest : {WIPE_BEFORE_INGEST}")
print(f"  Sync after ingest  : {SYNC_AFTER_INGEST}")

# List PDFs from the volume (used by the ingestion cell)
import os
pdf_files = sorted(f for f in os.listdir(VOLUME_PATH) if f.lower().endswith(".pdf"))
if not pdf_files:
    raise FileNotFoundError(f"No PDF files found at {VOLUME_PATH}. Upload .pdf files before running.")
print(f"  PDFs found         : {len(pdf_files)} ({', '.join(pdf_files[:5])}{'...' if len(pdf_files) > 5 else ''})")
print("=" * 55)

In [ ]:
# ================================================================
# CELL 3 — Clear Existing Chunks (Controlled by WIPE_BEFORE_INGEST)
# ================================================================
#
# PURPOSE:
#   Optionally deletes all existing rows from the policy_documents
#   table before the new chunks are written. This step exists to
#   prevent stale data from persisting when PDFs are updated or removed.
#
# THE STALE CHUNK PROBLEM:
#   Suppose "Lending_Policy_v1.pdf" was ingested previously and
#   produced 120 chunks. The file is then updated to v2, which
#   rewords several sections. If you ingest v2 with WIPE = False,
#   the table ends up with both old and new chunks for the same
#   document. The RAG agent will retrieve a mix of v1 and v2 content,
#   potentially returning contradictory or outdated policy answers
#   to loan officers — which is a compliance risk.
#
# WIPE_BEFORE_INGEST = True (recommended for most runs):
#   Deletes all rows before ingesting. Guarantees a clean reload.
#   After Cell 5 completes, the table will contain ONLY the current
#   versions of all PDFs in the volume.
#   Use this whenever updating existing documents.
#
# WIPE_BEFORE_INGEST = False (append mode):
#   Skips the DELETE. Safe ONLY when adding brand-new PDFs that
#   have never been ingested and you are certain no existing chunks
#   need to be replaced. Use with caution.
#
# NOTE ON DELTA DELETE PERFORMANCE:
#   Delta's DELETE operation is transactional and creates a new
#   version in the table's history. The data is not physically
#   removed immediately — it is marked as deleted and cleaned up
#   by VACUUM. This is fine for this use case.
# ================================================================

# Show row count before wipe so the operator can see what's being cleared.
before = spark.sql(f"SELECT COUNT(*) AS n FROM {TABLE_POLICY}").collect()[0]["n"]
print(f"Rows currently in table : {before}")

if WIPE_BEFORE_INGEST:
    # Transactional DELETE — marks all rows as deleted in the Delta log.
    # Creates a new Delta version (visible in table history).
    spark.sql(f"DELETE FROM {TABLE_POLICY}")
    after = spark.sql(f"SELECT COUNT(*) AS n FROM {TABLE_POLICY}").collect()[0]["n"]
    print(f"Rows after wipe         : {after}")
    print("Table cleared. Ready for fresh ingestion.")
else:
    # Append mode — no rows deleted. New chunks will be added on top.
    # Only use when adding brand-new PDFs with no overlap with existing content.
    print("WIPE_BEFORE_INGEST = False — skipping wipe, appending to existing chunks.")

In [ ]:
# ================================================================
# CELL 4 — Parse, Chunk, and Write PDFs to Delta
# ================================================================
#
# PURPOSE:
#   The core ingestion cell. For each PDF in the volume:
#     1. Extracts raw text page-by-page using pypdf
#     2. Detects section boundaries using multi-pattern heading detection
#     3. Splits each section into ≤MAX_CHUNK_CHARS chunks at sentence boundaries
#     4. Discards chunks shorter than MIN_CHUNK_CHARS (noise filtering)
#     5. Prefixes each chunk with its section heading for citation
#     6. Assigns a stable MD5-based doc_id per chunk
#   Then writes all chunks to the policy_documents Delta table in one batch.
#
# ── HEADING_PATTERNS ─────────────────────────────────────────────
#   Three regex patterns are tried in priority order on each line:
#
#   Pattern 1 — Numbered sections (highest priority):
#     Matches: "4.2 Loan Requirements", "A.1 Definitions", "1.1.1 Overview"
#     Format:  ^([A-Z0-9]{1,3}\.(?:[0-9]{1,3}\.?){0,3})\s+(.+)$
#     Why first: numbered headings are the most reliable structure signal
#     in formal policy documents and carry explicit hierarchy.
#
#   Pattern 2 — Named section keywords:
#     Matches: "SECTION IV: Eligibility", "ARTICLE 3 — Borrower Requirements"
#     Format:  ^(SECTION|ARTICLE|PART|CHAPTER)\s+([IVXLCDM]+|\d+|...)
#     Why second: these are unambiguous structural keywords in legal/policy docs.
#
#   Pattern 3 — All-caps lines (lowest priority):
#     Matches: "LOAN-TO-VALUE REQUIREMENTS", "UNDERWRITING GUIDELINES"
#     Format:  ^([A-Z][A-Z\s\-\/]{8,60})$
#     Why last: all-caps lines are a weaker signal — some PDFs use them for
#     emphasis rather than structure. Min length of 8 chars and max of 60
#     avoids matching single words or entire paragraphs.
#
# ── make_doc_id(doc_name, section, chunk_index) ───────────────────
#   Generates a stable, unique ID for each chunk using MD5.
#   Stability is important: if the same PDF is re-ingested with the
#   same content, the same chunks get the same IDs. This allows the
#   Vector Search delta-sync to detect unchanged rows and skip
#   re-embedding them (cheaper and faster incremental sync).
#   Format: MD5 hex digest of "{doc_name}::{section}::{chunk_index}"
#
# ── split_at_sentence_boundary(text, max_chars) ───────────────────
#   Splits a long section text into ≤max_chars chunks.
#   Algorithm:
#     1. Split the section text on sentence boundaries (. ! ?)
#     2. Greedily accumulate sentences until adding the next sentence
#        would exceed max_chars
#     3. Flush the current chunk and start a new one
#     4. If a single sentence is longer than max_chars (unusual but
#        possible in policy boilerplate), hard-split it character by
#        character as a fallback
#   This ensures chunks never cut a sentence in half, which preserves
#   the LLM's ability to reason over complete grammatical units.
#
# ── detect_sections(full_text) ────────────────────────────────────
#   Splits the full extracted PDF text into (heading, body) pairs.
#   Iterates line by line, testing each line against HEADING_PATTERNS.
#   When a heading is detected:
#     - The accumulated body text for the previous section is flushed
#     - A new section starts under the new heading
#   If no heading is ever detected (e.g. unstructured PDFs), falls back
#   to treating the entire document as one section under "Document".
#   The fallback ensures every PDF produces at least some chunks.
#
# ── process_pdf(pdf_path, doc_name) ──────────────────────────────
#   Top-level pipeline function called once per PDF file:
#     1. Opens the PDF with PdfReader and joins all pages' text
#     2. Calls detect_sections to get (heading, body) pairs
#     3. Calls split_at_sentence_boundary on each section body
#     4. Discards chunks below MIN_CHUNK_CHARS (noise filter)
#     5. Prepends the section heading to each chunk:
#        "[4.2 LTV Requirements] The maximum LTV ratio for..."
#        This prefix is critical — it flows through to the VS index,
#        so when the RAG agent retrieves a chunk, it immediately
#        knows which section to cite.
#     6. Assigns a stable doc_id and returns a list of Row objects
#
# ── Main ingestion loop ───────────────────────────────────────────
#   Calls process_pdf for each PDF, collects all Row objects into
#   all_rows, then writes in a single Spark DataFrame append.
#   Writing in one batch is more efficient than writing per-PDF
#   (one Delta transaction instead of N).
# ================================================================

import re
import hashlib
from pypdf import PdfReader
from pyspark.sql import Row

# ── Heading detection patterns — tried in priority order ─────────
# Each pattern is tested against each line of the extracted PDF text.
# The first pattern to match wins and the line is treated as a heading.
HEADING_PATTERNS = [
    # Priority 1: Numbered sections (e.g. "4.2", "4.2.1", "A.1 Definitions")
    re.compile(r'^([A-Z0-9]{1,3}\.(?:[0-9]{1,3}\.?){0,3})\s+(.+)$', re.MULTILINE),
    # Priority 2: Named section keywords (e.g. "SECTION IV:", "ARTICLE 3 —")
    re.compile(r'^(SECTION|ARTICLE|PART|CHAPTER)\s+([IVXLCDM]+|\d+|[A-Z]+)[:\s\-]+(.*)$', re.MULTILINE | re.IGNORECASE),
    # Priority 3: All-caps lines — weaker signal, used as fallback heading detection
    re.compile(r'^([A-Z][A-Z\s\-\/]{8,60})$', re.MULTILINE),
]

def make_doc_id(doc_name: str, section: str, chunk_index: int) -> str:
    """
    Generate a stable, unique ID for each chunk using MD5.

    Stability means the same chunk always gets the same ID across
    re-ingestion runs, enabling the VS delta-sync to detect unchanged
    rows and skip re-embedding them (incremental sync efficiency).

    Input: "{doc_name}::{section}::{chunk_index}"
    Output: 32-character hex MD5 digest
    """
    raw = f"{doc_name}::{section}::{chunk_index}"
    return hashlib.md5(raw.encode()).hexdigest()


def split_at_sentence_boundary(text: str, max_chars: int) -> list:
    """
    Split text into chunks of at most max_chars characters.

    Always splits at sentence boundaries (. ! ?) to preserve
    grammatical completeness. Never cuts a sentence in half.
    Hard-splits only as a last resort for pathologically long sentences.

    Args:
        text:      The section body text to split.
        max_chars: Maximum characters per output chunk (MAX_CHUNK_CHARS from config).

    Returns:
        List of non-empty string chunks, each ≤ max_chars.
    """
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    chunks, current = [], ""

    for sentence in sentences:
        if len(current) + len(sentence) + 1 <= max_chars:
            # Sentence fits in the current chunk — accumulate it.
            current = f"{current} {sentence}".strip()
        else:
            if current:
                chunks.append(current.strip())   # Flush current chunk
            if len(sentence) > max_chars:
                # Single sentence exceeds max_chars — hard-split as fallback.
                # Rare in well-formatted policy docs, common in boilerplate.
                for i in range(0, len(sentence), max_chars):
                    chunks.append(sentence[i:i + max_chars].strip())
                current = ""
            else:
                current = sentence   # Start a new chunk with this sentence

    if current:
        chunks.append(current.strip())   # Flush the final chunk

    # Filter out any empty strings that may have slipped through.
    return [c for c in chunks if c]


def split_lines_to_chunks_with_pages(line_pages: list, max_chars: int) -> list:
    """
    Split (line_text, page_num) pairs into text chunks at sentence boundaries.

    Each chunk gets the page number of its first sentence so page attribution
    is accurate even when a section spans multiple PDF pages.

    Returns:
        List of (chunk_text, page_num) tuples.
    """
    if not line_pages:
        return []

    # Build sentence-level items, tracking the page where each sentence starts
    sentences_with_pages = []
    pending_text = ""
    pending_page = None

    for line, page_num in line_pages:
        line = line.strip()
        if not line:
            continue
        if pending_page is None:
            pending_page = page_num

        combined = f"{pending_text} {line}".strip() if pending_text else line

        # Emit complete sentences (ending in . ! ?) as they become available
        parts = re.split(r'(?<=[.!?])\s+', combined)
        if len(parts) > 1:
            for part in parts[:-1]:
                if part.strip():
                    sentences_with_pages.append((part.strip(), pending_page))
            pending_text = parts[-1]
            pending_page = page_num  # Remainder sentence starts on this page
        else:
            pending_text = combined  # Sentence still accumulating; page unchanged

    if pending_text.strip():
        sentences_with_pages.append((pending_text.strip(), pending_page or 1))

    # Greedily merge sentences into chunks of at most max_chars
    result = []
    current_chunk = ""
    chunk_page = sentences_with_pages[0][1] if sentences_with_pages else 1

    for sentence, pnum in sentences_with_pages:
        if not current_chunk:
            chunk_page = pnum  # First sentence in a new chunk sets the page

        if len(current_chunk) + len(sentence) + 1 <= max_chars:
            current_chunk = f"{current_chunk} {sentence}".strip() if current_chunk else sentence
        else:
            if current_chunk:
                result.append((current_chunk.strip(), chunk_page))
            if len(sentence) > max_chars:
                # Hard-split as last resort for pathologically long sentences
                for i in range(0, len(sentence), max_chars):
                    result.append((sentence[i:i + max_chars].strip(), pnum))
                current_chunk = ""
            else:
                current_chunk = sentence
                chunk_page = pnum

    if current_chunk:
        result.append((current_chunk.strip(), chunk_page))

    return [(t, p) for t, p in result if t.strip()]


def process_pdf(pdf_path: str, doc_name: str) -> list:
    """
    Full pipeline for a single PDF: extract -> section -> chunk -> Row objects.

    Tracks page numbers at chunk level — each chunk gets the page its text
    actually starts on, not the page of the section heading. This is achieved
    by accumulating (line, page_num) tuples through the section detection step
    and passing them to split_lines_to_chunks_with_pages.

    Returns:
        List of Row objects ready for spark.createDataFrame().
    """
    reader = PdfReader(pdf_path)

    # Build (line, page_num) pairs with 1-based page numbers
    line_page_pairs = []
    for page_num, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        for line in text.split("\n"):
            line_page_pairs.append((line, page_num))

    # Section detection — accumulate (line, page_num) tuples per section
    sections = []  # list of (heading, [(line, page_num), ...])
    current_heading    = "Introduction"
    current_line_pages = []
    found_any_heading  = False

    for line, page_num in line_page_pairs:
        line_stripped = line.strip()
        if not line_stripped:
            continue

        matched_heading = None
        for pattern in HEADING_PATTERNS:
            if pattern.match(line_stripped):
                matched_heading = line_stripped[:HEADING_MAX_LENGTH]
                break

        if matched_heading:
            if current_line_pages:
                sections.append((current_heading, current_line_pages))
            current_heading    = matched_heading
            current_line_pages = []
            found_any_heading  = True
        else:
            current_line_pages.append((line_stripped, page_num))

    if current_line_pages:
        sections.append((current_heading, current_line_pages))

    if not found_any_heading:
        print("    No headings detected — using full-document fallback.")
        all_lp = [lp for _, lp_list in sections for lp in lp_list]
        sections = [("Document", all_lp)]

    rows        = []
    chunk_index = 0

    for section_heading, line_pages in sections:
        chunks_with_pages = split_lines_to_chunks_with_pages(line_pages, MAX_CHUNK_CHARS)
        for chunk_text, chunk_page in chunks_with_pages:
            if len(chunk_text.strip()) < MIN_CHUNK_CHARS:
                continue
            rows.append(Row(
                doc_id     = make_doc_id(doc_name, section_heading, chunk_index),
                doc_name   = doc_name,
                section    = section_heading,
                chunk_text = f"[{section_heading}] {chunk_text}",
                page_num   = chunk_page
            ))
            chunk_index += 1

    return rows

# ── Main ingestion loop ──────────────────────────────────────────
# Process all PDFs, collecting Row objects into a flat list.
# All rows are written in a single Spark DataFrame append at the end
# (one Delta transaction is more efficient than N per-file transactions).
all_rows = []
print(f"Processing {len(pdf_files)} PDF(s)...\n")

for filename in sorted(pdf_files):
    full_path = os.path.join(VOLUME_PATH, filename)
    print(f"  Processing: {filename}")
    rows = process_pdf(full_path, filename)
    all_rows.extend(rows)
    print(f"    → {len(rows)} chunks")

print(f"\nTotal chunks generated: {len(all_rows)}")

# ── Write to Delta in one batch ──────────────────────────────────
if not all_rows:
    raise ValueError(
        "No chunks generated. Check PDF content and heading patterns.\n"
        "If all PDFs triggered the fallback, check MIN_CHUNK_CHARS is not too high.\n"
        "If pypdf returns empty text, the PDF may be image-only and needs OCR first."
    )

# Add page_num column if this is the first run with page tracking
try:
    spark.sql(f"ALTER TABLE {TABLE_POLICY} ADD COLUMN page_num BIGINT")
    print("Added page_num column to policy_documents")
except Exception:
    pass  # Column already exists

# createDataFrame infers schema from Row field names — matches policy_documents DDL.
df = spark.createDataFrame(all_rows)
df.write.format("delta").mode("append").saveAsTable(TABLE_POLICY)

final_count = spark.sql(f"SELECT COUNT(*) AS n FROM {TABLE_POLICY}").collect()[0]["n"]
print(f"Rows written to {TABLE_POLICY}: {final_count}")
print("Write complete. Proceed to Cell 6 (VS sync).")

In [ ]:
# ================================================================
# CELL 5 — Trigger Vector Search Index Sync
# ================================================================
#
# PURPOSE:
#   Triggers a re-sync of the Vector Search index against the
#   policy_documents Delta table. This is the step that turns the
#   raw text chunks written in Cell 5 into searchable embedding
#   vectors that the RAG agent can query at inference time.
#
# WHY A SYNC IS NEEDED:
#   The VS index does not automatically update when rows are written
#   to the source Delta table. This is because the index uses a
#   TRIGGERED pipeline (set in 01_infrastructure Cell 8) — it only
#   re-embeds when explicitly instructed to. This design is deliberate:
#   it avoids continuous background compute cost for a document set
#   that changes infrequently (policy documents are updated quarterly,
#   not continuously).
#
# WHAT THE SYNC DOES:
#   1. Reads the Change Data Feed on policy_documents since the last sync
#   2. Sends new/changed chunk_text values to the embedding model
#      (databricks-gte-large-en) to generate vector representations
#   3. Updates the ANN index with the new vectors
#   4. Marks deleted rows as removed from the index
#   Only changed rows are re-embedded — this is an incremental sync,
#   not a full re-index. For a typical update (new PDF added), this
#   usually completes in 2–5 minutes.
#
# SYNC_AFTER_INGEST = True (default, recommended):
#   Triggers the sync immediately after the Delta write. The RAG agent
#   will return results from the new chunks after sync completes.
#
# SYNC_AFTER_INGEST = False:
#   Skips the sync. Use ONLY when batching multiple ingestion runs
#   back-to-back (e.g. loading 10 PDFs one at a time) and you want
#   to trigger a single sync manually at the end to save time.
#   To trigger manually: run vsc.get_index(...).sync() in a new cell.
#
# POLLING:
#   Sync is asynchronous — this cell polls every 20 seconds and
#   blocks until the index reports ready=True with state
#   "ONLINE_TRIGGERED_UPDATE" (meaning: online and the most recent
#   triggered update has completed). Exits immediately on FAILED state.
#
# TIMING:
#   Typically 2–5 minutes for a small update.
#   Can take longer (up to 15 min) for the first sync after a full wipe.
# ================================================================

import time
from databricks.vector_search.client import VectorSearchClient

# disable_notice=True suppresses the "client created" log line.
vsc = VectorSearchClient(disable_notice=True)

if not SYNC_AFTER_INGEST:
    # Sync skipped — the RAG agent will NOT see new chunks until a sync is triggered.
    print("SYNC_AFTER_INGEST = False — skipping sync.")
    print("Remember to trigger a manual sync before using the RAG agent.")
    print("Run: vsc.get_index(VS_ENDPOINT_NAME, VS_INDEX_NAME).sync()")
else:
    print("Triggering Vector Search index sync...")
    # .sync() is non-blocking — it starts the sync pipeline and returns immediately.
    # The polling loop below tracks completion.
    vsc.get_index(VS_ENDPOINT_NAME, VS_INDEX_NAME).sync()

    print("Polling for sync completion (checking every 20s)...")
    while True:
        idx        = vsc.get_index(VS_ENDPOINT_NAME, VS_INDEX_NAME)
        idx_status = idx.describe().get("status", {})
        detailed   = idx_status.get("detailed_state", "")  # e.g. "SYNCING", "ONLINE_TRIGGERED_UPDATE"
        ready      = idx_status.get("ready", False)
        print(f"  {detailed}  |  ready={ready}")

        if ready and detailed == "ONLINE_TRIGGERED_UPDATE":
            # Index is online and the latest triggered sync is complete.
            # New chunks are now searchable by the RAG agent.
            print("Sync complete. Index is up to date.")
            break
        elif "FAILED" in detailed.upper():
            # Sync failed — surface the state immediately rather than polling forever.
            raise RuntimeError(f"Sync failed with state: {detailed}")

        time.sleep(20)